In [1]:
import numpy as np
import jax.numpy as jnp
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

import hj_reachability as hj
from hj_reachability import sets
from hj_reachability.systems.doubleint import DoubleInt
from hj_reachability.zg_solver import ZGSolverSettings, step_until_converged

In [2]:
import os
# =========================
# Save path
# =========================


# ell + u_max dir: 
save_dir = Path("/home/zg0327/projects/HJR/hj_reachability/examples/Data/DoubleInt/clvf_ell_umax")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "doubleint_clvf_ell_umax_neuralop.npz"

print("save_path:", save_path)

save_path: /home/zg0327/projects/HJR/hj_reachability/examples/Data/DoubleInt/clvf_ell_umax/doubleint_clvf_ell_umax_neuralop.npz


In [3]:
# =========================
# Grid settings
# =========================

x1_min, x1_max = -2.0, 2.0
x2_min, x2_max = -2.0, 2.0
H, W = 101, 101

grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    sets.Box(
        np.array([x1_min, x2_min]),
        np.array([x1_max, x2_max])
    ),
    (H, W)
)

x1s = np.asarray(grid.coordinate_vectors[0], dtype=np.float32)
x2s = np.asarray(grid.coordinate_vectors[1], dtype=np.float32)

X1, X2 = np.meshgrid(x1s, x2s, indexing="ij")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [14]:
# =========================
# Parameters
# =========================

# gamma fixed
gamma_fixed  = np.float32(0.3)

# ell parameters, same as before
a_list = np.arange(0.1, 3.0 + 1e-9, 0.1).astype(np.float32)
b_list = np.array([1.0], dtype=np.float32)

# new parameter: u_max
u_max_list = np.arange(1, 2.0 + 1e-9, 0.1).astype(np.float32)

# disturbance fixed
d1_max = np.float32(0.0)
d2_max = np.float32(0.0)


print("gamma_fixed:", gamma_fixed)
print("a_list:", a_list.shape, a_list[0], a_list[-1])
print("b_list:", b_list)
print("u_max_list:", u_max_list.shape, u_max_list[0], u_max_list[-1])

gamma_fixed: 0.3
a_list: (30,) 0.1 3.0
b_list: [1.]
u_max_list: (11,) 1.0 2.0


In [15]:
# =========================
# Loss function
# =========================

def make_ell_Q(a, b):
    """
    ell_Q(x) = sqrt(a*x1^2 + b*x2^2) - target_radius
    """
    ell = np.sqrt(a * X1**2 + b * X2**2) 
    return ell.astype(np.float32)

# =========================
# Solver settings
# =========================

initial_time = 0.0
target_time = -20.0

convergence_threshold = 1e-4
divergence_threshold = 20

def compute_clvf_one_sample(loss_values, gamma, u_max):
    """
    Compute CLVF for one sample:
        fixed gamma,
        one u_max,
        one loss ell(x; a, b).
    """

    dynamics = DoubleInt(
        u_max=float(u_max),
        d1_max=float(d1_max),
        d2_max=float(d2_max),
        gamma=float(gamma),
        control_mode="min",
        disturbance_mode="max",
    )

    solver_settings = ZGSolverSettings(
        convergence_threshold=convergence_threshold,
        divergence_threshold=divergence_threshold,
        value_postprocessor=hj.solver.static_obstacle(loss_values),
    )

    values0 = jnp.asarray(loss_values)

    V = step_until_converged(
        solver_settings=solver_settings,
        dynamics=dynamics,
        grid=grid,
        time=initial_time,
        values=values0,
        target_time=target_time,
        convergence_threshold=convergence_threshold,
        progress_bar=True,
    )

    return np.asarray(V, dtype=np.float32)

In [ ]:
# =========================
# Main loop over a, b, u_max
# =========================

ells = []
Vs = []

a_all = []
b_all = []
u_max_all = []
gamma_all = []
Qs = []

total_num = len(a_list) * len(b_list) * len(u_max_list)

sample_id = 0

for a in a_list:
    for b in b_list:
        ell = make_ell_Q(
            a=a,
            b=b,
        )

        Q = np.array(
            [
                [a, 0.0],
                [0.0, b],
            ],
            dtype=np.float32,
        )

        for u_max in u_max_list:
            sample_id += 1

            print(
                f"\nSample {sample_id}/{total_num}: "
                # f"a={a:.2f}, b={b:.2f}, u_max={u_max:.2f}, gamma={gamma_fixed:.2f}"
            )

            V = compute_clvf_one_sample(
                loss_values=ell,
                gamma=gamma_fixed,
                u_max=u_max,
            )

            ell_np = np.asarray(ell, dtype=np.float32)
            V_np = np.asarray(V, dtype=np.float32)

            if ell_np.shape != (H, W):
                raise ValueError(f"ell shape mismatch: got {ell_np.shape}, expected {(H, W)}")

            if V_np.shape != (H, W):
                raise ValueError(f"V shape mismatch: got {V_np.shape}, expected {(H, W)}")

            ells.append(ell_np)
            Vs.append(V_np)

            a_all.append(np.float32(a))
            b_all.append(np.float32(b))
            u_max_all.append(np.float32(u_max))
            gamma_all.append(np.float32(gamma_fixed))
            Qs.append(Q)

ells = np.stack(ells, axis=0).astype(np.float32)              # [N, H, W]
Vs = np.stack(Vs, axis=0).astype(np.float32)                  # [N, H, W]

a_all = np.array(a_all, dtype=np.float32)                     # [N]
b_all = np.array(b_all, dtype=np.float32)                     # [N]
u_max_all = np.array(u_max_all, dtype=np.float32)             # [N]
gamma_all = np.array(gamma_all, dtype=np.float32)             # [N]
Qs = np.stack(Qs, axis=0).astype(np.float32)                  # [N, 2, 2]

print("ells:", ells.shape)
print("Vs:", Vs.shape)
print("a_all:", a_all.shape)
print("b_all:", b_all.shape)
print("u_max_all:", u_max_all.shape)
print("gamma_all:", gamma_all.shape)
print("Qs:", Qs.shape)


Sample 1/330: 


100%|##########| 20.0000/20.0 [00:00<00:00, 29.12sim_s/s]



Sample 2/330: 


100%|##########| 20.0000/20.0 [-00:00<00:00, -67.68sim_s/s]



Sample 3/330: 


...


Sample 329/330: 


 15%|#5        |  3.0154/20.0 [00:00<00:00, 22.98sim_s/s]



Sample 330/330: 


 24%|##3       |  4.7625/20.0 [00:00<00:00, 22.25sim_s/s]

ells: (330, 101, 101)
Vs: (330, 101, 101)
a_all: (330,)
b_all: (330,)
u_max_all: (330,)
gamma_all: (330,)
Qs: (330, 2, 2)


In [20]:
# =========================
# Build neuralop-ready data
# =========================

N = ells.shape[0]

x_ell = ells[:, None, :, :]      # [N, 1, H, W]

u_max_channel = u_max_all[:, None, None, None] * np.ones(
    (N, 1, H, W),
    dtype=np.float32
)                                # [N, 1, H, W]

x_data = np.concatenate(
    [x_ell, u_max_channel],
    axis=1
).astype(np.float32)             # [N, 2, H, W]

y_data = Vs[:, None, :, :].astype(np.float32)   # [N, 1, H, W]

print("x_data:", x_data.shape)
print("y_data:", y_data.shape)
print("x_data dtype:", x_data.dtype)
print("y_data dtype:", y_data.dtype)

x_data: (330, 2, 101, 101)
y_data: (330, 1, 101, 101)
x_data dtype: float32
y_data dtype: float32


In [21]:
np.savez_compressed(
    save_path,

    # Neuralop-ready tensors as numpy arrays
    x_data=x_data,              # [N, 2, H, W], input = [ell_Q, u_max]
    y_data=y_data,              # [N, 1, H, W], output = V

    # Raw data
    ells=ells,                  # [N, H, W]
    Vs=Vs,                      # [N, H, W]

    # Metadata for each sample
    a_values=a_all,             # [N]
    b_values=b_all,             # [N]
    u_max_values=u_max_all,     # [N]
    gammas=gamma_all,           # [N], fixed but repeated
    Qs=Qs,                      # [N, 2, 2]

    # Parameter lists
    a_list=a_list,
    b_list=b_list,
    u_max_list=u_max_list,
    gamma_fixed=np.float32(gamma_fixed),

    # Grid
    x1s=x1s,                    # [H]
    x2s=x2s,                    # [W]

    # Experiment settings
    d1_max=np.float32(d1_max),
    d2_max=np.float32(d2_max),
    target_radius=np.float32(target_radius),
    initial_time=np.float32(initial_time),
    target_time=np.float32(target_time),
    convergence_threshold=np.float32(convergence_threshold),
    divergence_threshold=np.float32(divergence_threshold),

    # Channel names
    input_channels=np.array(["ell_Q", "u_max"]),
    output_channels=np.array(["V"]),
    system_name=np.array("DoubleInt"),
)

print("Saved to:")
print(save_path)

Saved to:
/home/zg0327/projects/HJR/hj_reachability/examples/Data/DoubleInt/clvf_ell_umax/doubleint_clvf_ell_umax_neuralop.npz
